In [ ]:
# =========================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN & XÁC THỰC GCP
# =========================================================
!pip install pyspark gcsfs -q

from google.colab import auth
auth.authenticate_user()
print("Xác thực GCP thành công!")

✅ Xác thực GCP thành công!


In [ ]:
# =============================================================
# CELL 2 — KHỞI TẠO SPARKSESSION
# =============================================================
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import gcsfs
import gc
import time
import os
import signal
import subprocess

# Dừng session cũ nếu còn tồn tại
try:
    if 'spark' in vars() and spark is not None:
        spark.stop()
        print(' Đã dừng SparkSession cũ.')
except BaseException:
    pass

def _build_spark() -> SparkSession:
    """Tạo SparkSession với cấu hình tối ưu cho Colab T4 (~12 GB RAM)."""
    return (
        SparkSession.builder
        .appName('AmazonReviews_Silver_Pipeline')
        .config('spark.jars.packages',
                'com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.5')
        .config('spark.hadoop.fs.gs.impl',
                'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem')
        .config('spark.hadoop.fs.AbstractFileSystem.gs.impl',
                'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS')
        .config('spark.hadoop.google.cloud.auth.service.account.enable', 'true')
        .config('spark.sql.caseSensitive', 'true')
        .config('spark.driver.memory',          '6g')
        .config('spark.executor.memory',        '6g')
        .config('spark.memory.offHeap.enabled', 'true')
        .config('spark.memory.offHeap.size',    '2g')
        .config('spark.driver.maxResultSize',   '2g')
        .config('spark.sql.shuffle.partitions', '20')
        .config("spark.rapids.sql.concurrentGpuTasks", "2") # Cho phép 2 task GPU chạy song song
        .config("spark.driver.memory", "12g")              # Tăng RAM driver để gánh 2 luồng
        .config("spark.executor.memory", "12g")
        .getOrCreate()
    )

spark = _build_spark()
spark.sparkContext.setLogLevel('ERROR')
print(f'SparkSession khởi tạo thành công! (version {spark.version})')

✅ SparkSession khởi tạo thành công! (version 4.0.2)


In [ ]:
# =============================================================
# CELL 3 — HÀM RESTART SPARK KHI JVM CRASH
# =============================================================

def _force_kill_java() -> None:
    """Tìm và kill tất cả process 'java' đang chạy trong Colab VM."""
    try:
        result = subprocess.run(
            ['pgrep', '-f', 'pyspark'],
            capture_output=True, text=True
        )
        pids = [int(p) for p in result.stdout.strip().split() if p.isdigit()]
        for pid in pids:
            try:
                os.kill(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass  
        if pids:
            print(f' Đã kill {len(pids)} JVM process: {pids}')
    except BaseException:
        pass  


def restart_spark(wait: int = 12) -> None:
    """
    Dừng SparkSession hiện tại, force-kill JVM còn sót, chờ `wait` giây,
    rồi khởi động lại session mới. Cập nhật biến toàn cục `spark`.

    FIX v4:
      - Dùng `except BaseException` thay vì `except Exception` để bắt
        Py4JNetworkError (kế thừa BaseException trực tiếp)
      - Gọi _force_kill_java() để đảm bảo cổng JVM được giải phóng
        trước khi _build_spark() chạy
    """
    global spark
    print(f'Đang restart SparkSession (chờ {wait}s)...')

    try:
        spark.stop()
    except BaseException:  
        pass

    _force_kill_java()

    time.sleep(wait)
    gc.collect()

    spark = _build_spark()
    spark.sparkContext.setLogLevel('ERROR')
    print('SparkSession đã được khởi động lại!')


def is_jvm_crash(exc: BaseException) -> bool:
    """
    Trả True nếu exception là do JVM/py4j crash.
    Kiểm tra cả tên class lẫn message vì Py4J có nhiều subclass.
    """
    crash_types = {
        'ConnectionRefusedError',
        'BrokenPipeError',
        'Py4JNetworkError',
        'Py4JJavaError',
        'Py4JError',
    }
    crash_msgs = [
        'connection refused',
        'answer from java side is empty',
        'java gateway process exited',
        'broken pipe',
        'an error occurred while calling',
    ]
    type_name = type(exc).__name__
    msg       = str(exc).lower()

    return (
        type_name in crash_types
        or any(s in msg for s in crash_msgs)
    )


def spark_safe_clear_cache() -> None:
    """Gọi spark.catalog.clearCache() an toàn — bỏ qua nếu JVM đã chết."""
    try:
        spark.catalog.clearCache()
    except BaseException:   # <-- FIX: bảo vệ finally block
        pass


def spark_safe_unpersist(df) -> None:
    """Gọi df.unpersist() an toàn — bỏ qua nếu JVM đã chết."""
    try:
        if df is not None:
            df.unpersist()
    except BaseException:
        pass


print('restart_spark(), is_jvm_crash(), spark_safe_clear_cache() đã sẵn sàng.')

✅ restart_spark(), is_jvm_crash(), spark_safe_clear_cache() đã sẵn sàng.


In [ ]:
# =============================================================
# CELL 4 — CẤU HÌNH ĐƯỜNG DẪN GCS & QUÉT DANH MỤC
# =============================================================
fs = gcsfs.GCSFileSystem()

BUCKET         = 'amazon-reviews-lakehouse-data'
bronze_base    = f'{BUCKET}/bronze-zone'
silver_base    = f'{BUCKET}/silver-zone'
NUM_PARTITIONS = 20    # Chia nhỏ để mỗi task dùng ít RAM hơn


def scan_folders(gcs_path: str) -> list:
    """Quét và trả về danh sách tên thư mục hợp lệ trên GCS."""
    try:
        entries = fs.ls(gcs_path)
    except FileNotFoundError:
        print(f'Không tìm thấy: {gcs_path}')
        return []
    folders = [e.split('/')[-1] for e in entries if fs.isdir(e)]
    return [
        f for f in folders
        if f and not f.startswith('_') and not f.startswith('.')
    ]


meta_folders   = scan_folders(f'{bronze_base}/meta-data/')
review_folders = scan_folders(f'{bronze_base}/review-data/')

print(f'Meta   : {len(meta_folders):>3} danh mục — {meta_folders}')
print(f'Review : {len(review_folders):>3} danh mục — {review_folders}')

✅ Meta   :  31 danh mục — ['All_Beauty', 'Amazon_Fashion', 'Appliances', 'Arts_Crafts_and_Sewing', 'Automotive', 'Books', 'CDs_and_Vinyl', 'Cell_Phones_and_Accessories', 'Clothing_Shoes_and_Jewelry', 'Digital_Music', 'Electronics', 'Gift_Cards', 'Grocery_and_Gourmet_Food', 'Health_and_Household', 'Health_and_Personal_Care', 'Home_and_Kitchen', 'Industrial_and_Scientific', 'Kindle_Store', 'Magazine_Subscriptions', 'Movies_and_TV', 'Musical_Instruments', 'Office_Products', 'Patio_Lawn_and_Garden', 'Pet_Supplies', 'Software', 'Sports_and_Outdoors', 'Subscription_Boxes', 'Tools_and_Home_Improvement', 'Toys_and_Games', 'Unknown', 'Video_Games']
✅ Review :  31 danh mục — ['All_Beauty', 'Amazon_Fashion', 'Appliances', 'Arts_Crafts_and_Sewing', 'Automotive', 'Books', 'CDs_and_Vinyl', 'Cell_Phones_and_Accessories', 'Clothing_Shoes_and_Jewelry', 'Digital_Music', 'Electronics', 'Gift_Cards', 'Grocery_and_Gourmet_Food', 'Health_and_Household', 'Health_and_Personal_Care', 'Home_and_Kitchen', 'Indus

In [5]:
# =============================================================
# CELL 5 — HÀM TIỆN ÍCH DÙNG CHUNG
# =============================================================
_JUNK = ('', 'null', 'NULL', 'N/A', 'n/a', 'None', 'none', 'NaN', 'nan')


def clean_string_columns(df):
    """Trim + chuẩn hoá giá trị rác cho tất cả cột string."""
    for col_name, dtype in df.dtypes:
        if dtype == 'string':
            df = df.withColumn(col_name, F.trim(F.col(col_name)))
            df = df.withColumn(
                col_name,
                F.when(F.col(col_name).isin(*_JUNK), F.lit(None))
                 .otherwise(F.col(col_name))
            )
    return df


def safe_cast(df, col_name: str, target_type: str):
    """Cast an toàn: chuẩn hoá rác → NULL trước, rồi mới cast kiểu."""
    raw = F.trim(F.col(col_name).cast('string'))
    return df.withColumn(
        col_name,
        F.when(raw.isin(*_JUNK), F.lit(None))
         .otherwise(F.col(col_name))
         .cast(target_type)
    )


print('✅ clean_string_columns() và safe_cast() đã sẵn sàng.')

✅ clean_string_columns() và safe_cast() đã sẵn sàng.


In [ ]:
# =============================================================
# CELL 6 & 7 — HÀM TIỀN XỬ LÝ
# =============================================================

def process_review_silver(df, cat_name):
    # Gắn trực tiếp tên danh mục từ vòng lặp ngoài
    df = df.withColumn("source_category", F.lit(cat_name))

    # 1. Quét rác toàn cục
    df = clean_string_columns(df)

    # 2. Ép kiểu an toàn (Bắt buộc dùng biến trung gian dạng string để tránh bẫy kiểu)
    raw_ts = F.trim(F.col("timestamp_raw").cast("string"))
    df = df.withColumnRenamed("timestamp", "timestamp_raw") \
           .withColumn("timestamp", F.to_timestamp(F.when(raw_ts.isin("", "null"), F.lit(None)).otherwise(F.col("timestamp_raw")) / 1000))

    raw_rating = F.trim(F.col("rating").cast("string"))
    df = df.withColumn("rating", F.when(raw_rating.isin("", "null"), F.lit(None)).otherwise(F.col("rating")).cast("byte"))

    raw_vp = F.trim(F.col("verified_purchase").cast("string"))
    df = df.withColumn("verified_purchase", F.when(raw_vp.isin("", "null"), F.lit(None)).otherwise(F.col("verified_purchase")).cast("boolean"))

    raw_hv = F.trim(F.col("helpful_vote").cast("string"))
    df = df.withColumn("helpful_vote", F.when(raw_hv.isin("", "null"), F.lit(None)).otherwise(F.col("helpful_vote")).cast("int"))

    # 3. Lọc bỏ các dòng thiếu dữ liệu cốt lõi
    review_drop_cols = ["user_id", "parent_asin", "rating", "timestamp", "verified_purchase", "helpful_vote"]
    df = df.dropna(subset=[c for c in review_drop_cols if c in df.columns])

    # 4. Làm sạch văn bản bằng Regex
    df = df.withColumn("text", F.regexp_replace(F.col("text").cast("string"), r"<[^>]*>", " ")) \
           .withColumn("text", F.regexp_replace(F.col("text"), r"https?://\S+|www\.\S+", "")) \
           .withColumn("text", F.regexp_replace(F.col("text"), r"[^a-zA-Z0-9\s]", "")) \
           .withColumn("text", F.trim(F.regexp_replace(F.col("text"), r"\s+", " ")))

    # 5. Loại trùng lặp
    window_dedup = Window.partitionBy("user_id", "parent_asin", "timestamp_raw").orderBy(F.col("helpful_vote").desc_nulls_last())
    df = df.withColumn("rn", F.row_number().over(window_dedup)).filter(F.col("rn") == 1).drop("rn")

    df = df.withColumn("processed_at", F.current_timestamp())
    core_columns = ["user_id", "parent_asin", "rating", "text", "timestamp", "timestamp_raw", "verified_purchase", "helpful_vote", "source_category", "processed_at"]
    return df.select(*[c for c in core_columns if c in df.columns])


def process_meta_silver(df, cat_name):
    df = df.withColumn("source_category", F.lit(cat_name))

    df = clean_string_columns(df)

    meta_drop_cols = ["parent_asin"]

    if "average_rating" in df.columns:
        is_empty_rating = F.trim(F.col("average_rating").cast("string")).isin("", "null", "N/A")
        df = df.withColumn("average_rating",
                           F.when(is_empty_rating, F.lit(None)).otherwise(F.col("average_rating")).cast("float"))

    if "rating_number" in df.columns:
        is_empty_num = F.trim(F.col("rating_number").cast("string")).isin("", "null", "N/A")
        df = df.withColumn("rating_number",
                           F.when(is_empty_num, F.lit(None)).otherwise(F.col("rating_number")).cast("int"))

    if "price" in df.columns:
        str_price = F.col("price").cast("string")
        extracted_price = F.regexp_extract(str_price, r"(\d+\.\d+|\d+)", 1)
        df = df.withColumn("price",
                           F.when(extracted_price == "", F.lit(None)).otherwise(extracted_price).cast("float"))

    df = df.dropna(subset=meta_drop_cols)

    array_cols = ["categories", "features", "description"]
    for c in array_cols:
        if c in df.columns:
            df = df.withColumn(c, F.when(F.col(c).isNull(), F.array()).otherwise(F.col(c)))

    window_meta = Window.partitionBy("parent_asin").orderBy(F.col("parent_asin"))
    df = df.withColumn("rn", F.row_number().over(window_meta)).filter(F.col("rn") == 1).drop("rn")

    df = df.withColumn("processed_at", F.current_timestamp())
    return df

In [ ]:
# =============================================================
# CELL 8 — PIPELINE ETL CHÍNH (Đã vá lỗi Retry và Lazy Eval)
# =============================================================

def execute_silver_pipeline_safe(categories, sub_folder, data_type, max_retry=1):
    prefix = f"[{sub_folder.upper()}]"
    print(f"\n==============================================================")
    print(f"🚀 ETL BẮT ĐẦU: {sub_folder}  ({len(categories)} danh mục)")
    print(f"==============================================================")

    result = {"success": 0, "skipped": 0, "failed": []}

    for idx, cat in enumerate(categories, 1):
        source_path = f"gs://{bronze_base}/{sub_folder}/{cat}/*.parquet"
        target_path = f"gs://{silver_base}/{sub_folder}/{cat}"

        # Bỏ qua nếu đã làm xong
        if fs.exists(f"{target_path}/_SUCCESS"):
            print(f"  ⏩ [{idx:02d}/{len(categories)}] {cat} — Bỏ qua (đã có _SUCCESS)")
            result["skipped"] += 1
            continue

        print(f"⏳ [{idx:02d}/{len(categories)}] {cat} — Đang xử lý...")
        attempt = 0

        while attempt <= max_retry:
            df_bronze = None
            df_silver = None

            try:
                df_bronze = spark.read.parquet(source_path)
                df_bronze = df_bronze.repartition(10)

                if data_type == "Review":
                    df_silver = process_review_silver(df_bronze, cat)
                else:
                    df_silver = process_meta_silver(df_bronze, cat)

                df_silver.write.mode("overwrite")\
                    .option("compression", "gzip") \
                    .parquet(target_path)
                count_out = spark.read.parquet(target_path).count()

                print(f"  ✅ Thành công: {cat} | Đầu ra: {count_out:,} hàng")
                result["success"] += 1
                break

            except BaseException as exc:
                last_err = str(exc).split('\n')[0][:250]

                if is_jvm_crash(exc) and attempt < max_retry:
                    attempt += 1
                    print(f"  💥 JVM crash tại {cat} (Thử lại {attempt}/{max_retry}) — Đang restart Spark...")
                    restart_spark(wait=12)
                    continue 
                else:
                    print(f"  ❌ LỖI tại {cat}: {last_err}")
                    result["failed"].append(cat)
                    break 

            finally:
                if df_bronze is not None:
                    try: df_bronze.unpersist()
                    except: pass
                if df_silver is not None:
                    try: df_silver.unpersist()
                    except: pass
                try: spark.catalog.clearCache()
                except: pass
                gc.collect()

    # Báo cáo tổng kết nhánh
    print(f"\n==============================================================")
    print(f"🎉 HOÀN THÀNH: {sub_folder} ({data_type})")
    print(f"   ✅ Thành công : {result['success']}")
    print(f"   ⏩ Bỏ qua    : {result['skipped']}")
    print(f"   ❌ Lỗi       : {len(result['failed'])}")
    if result["failed"]:
        print(f"\n⚠️  Danh mục bị lỗi:")
        for err_cat in result["failed"]:
            print(f"   • {err_cat}")
    print(f"==============================================================")

    return result


In [ ]:
# nhẹ < 2 GB:
#low_group = [
#     'Subscription_Boxes', 'Magazine_Subscriptions', 'Gift_Cards', 'Digital_Music',
#     'Health_and_Personal_Care', 'All_Beauty', 'Appliances', 'Amazon_Fashion', 'Musical_Instruments', 'Software'
# ]

# vừa: [2 GB - 8 GB]
# medium_group = [
#     'Industrial_and_Scientific', 'Video_Games', 'CDs_and_Vinyl',
#     'Arts_Crafts_and_Sewing', 'Office_Products', 'Grocery_and_Gourmet_Food',
#     'Toys_and_Games', 'Patio_Lawn_and_Garden'
# ]

# [8 GB - 15 GB]
# high_group = [
#     'Pet_Supplies', 'Movies_and_TV', 'Automotive', 'Sports_and_Outdoors',
#     'Cell_Phones_and_Accessories', 'Health_and_Household', 'Tools_and_Home_Improvement'
# ]

# > 15 GB
# extreme_group = [
#     'Kindle_Store', 'Books', 'Electronics',
#     'Clothing_Shoes_and_Jewelry', 'Unknown', 'Home_and_Kitchen'
# ]

In [ ]:
# =============================================================
# CELL 9 — CHẠY TOÀN BỘ PIPELINE
# =============================================================
print('\n' + '#'*62)
print('#  AMAZON REVIEWS — BRONZE → SILVER PIPELINE')
print('#'*62)

meta_result = execute_silver_pipeline_safe(
    meta_folders, 'meta-data', 'Meta', max_retry=1
)

review_result = execute_silver_pipeline_safe(
    review_folders, 'review-data', 'Review', max_retry=1
)

# Tổng hợp
total_ok   = meta_result['success'] + review_result['success']
total_skip = meta_result['skipped'] + review_result['skipped']
total_fail = len(meta_result['failed']) + len(review_result['failed'])

print('\n' + '#'*62)
print('#  KẾT QUẢ TỔNG HỢP')
print('#'*62)
print(f' Tổng thành công : {total_ok}')
print(f' Tổng bỏ qua    : {total_skip}')
print(f' Tổng lỗi       : {total_fail}')

if total_fail == 0:
    print('\n Pipeline hoàn thành KHÔNG lỗi!')
else:
    all_failed = meta_result['failed'] + review_result['failed']
    print(f'\n  {total_fail} danh mục cần chạy lại:')
    for name, err in all_failed:
        print(f'   • {name}: {err}')